In [17]:
import pandas as pd

fluxes = pd.read_csv('fluxes.csv')
fluxes = fluxes.set_index("Var1")
fluxes.std(axis=1).to_csv('sd.csv')

fluxes.loc['PGIc_f']

Var2_1       0.705746
Var2_2       0.522159
Var2_3       0.820110
Var2_4       0.971271
Var2_5       0.906513
               ...   
Var2_7996    0.452744
Var2_7997    0.074555
Var2_7998    0.317464
Var2_7999    0.361057
Var2_8000    0.391794
Name: PGIc_f, Length: 8000, dtype: float64

In [21]:
def expand_rxn(rxn, label="id", integerize=True):

    def lab(m): return m if label == "obj" else getattr(m, label)

    reactants, products = [], []
    for met, coeff in rxn.metabolites.items():
        n = abs(coeff)
        n_int = int(round(n)) if integerize else 1
        if coeff < 0:
            reactants.extend([lab(met)] * n_int)
        elif coeff > 0:
            products.extend([lab(met)] * n_int)
    return reactants, products

In [22]:
import pandas as pd
from cobra.io import read_sbml_model

chlamymodel = read_sbml_model("photo_model_last.xml")

columns = ['#reaction_ID', 'reactant_IDs(atom)', 'product_IDs(atom)', 'reversibility']
df = pd.DataFrame(
    [[rxn.id, expand_rxn(rxn, label="id")[0], expand_rxn(rxn, label="id")[1], rxn.reversibility]
     for rxn in chlamymodel.reactions],
    columns=columns
)

No objective in listOfObjectives
No objective coefficients in model. Unclear what should be optimized


In [23]:
df.to_csv('metabolites.csv')

In [24]:
import re
import string
from collections import defaultdict, Counter, deque

MAPPING_FILE = "all_mapping.C.sorted.txt"
OUTPUT_FILE = "modreactionz.tsv"

def bname(spec): 
    return spec.split("[")[0]
def comp(spec):  
    return spec.split("[")[1].split("]")[0]

def pretty(spec, suffix=""):
    base = bname(spec).replace("_", "")
    if base and base[0].isdigit():
        base = "n" + base
    return f"{base}{comp(spec).upper()}{suffix}"

# parse mapping file
edges_all = []
by_rxn = defaultdict(list)
with open(MAPPING_FILE) as fh:
    for line in fh:
        m = pat.match(line.strip())
        if not m:
            continue
        rxn  = m.group("rxn")
        Lspec, Lidx = m.group("Lspec"), int(m.group("Lidx"))
        Rspec, Ridx = m.group("Rspec"), int(m.group("Ridx"))
        edges_all.append((rxn, (Lspec, Lidx), (Rspec, Ridx)))
        by_rxn[rxn].append(((Lspec, Lidx), (Rspec, Ridx)))

In [25]:
ALL_LETTERS   = list(string.ascii_lowercase)
UNIQUE        = [ch for ch in ALL_LETTERS if ch != RESERVED_MASK_LETTER]

def build_labels_basic(pairs):
    """One label per reactant C#; propagate to products. CoA masking when >26 reactant carbons."""
    if not pairs:
        return {}
    reactants  = {f"{Ls}:C#{Li}" for (Ls, Li), _ in pairs}
    labels     = defaultdict(dict)
    used       = set()
    for node in sorted(reactants, key=lambda x: (x.split(":")[0], int(x.split("#")[1]))):
        spec = node.split(":")[0]
        idx  = int(node.split("#")[1])

        ltr = next(ch for ch in UNIQUE if ch not in used)
        used.add(ltr)
        
        labels[spec][idx] = ltr
    for (Ls, Li), (Rs, Ri) in pairs:
        labels[Rs][Ri] = labels[Ls][Li]
    return labels


def build_labels_with_duplicates(pairs, left_specs, right_specs):
    """
    Duplicate-safe assignment that preserves product order exactly.
    Returns L_occ_letters, R_occ_letters, L_max, R_max.
    """
    left_mult  = Counter(left_specs)
    right_mult = Counter(right_specs)

    prod_targets = defaultdict(list)  # (Ls,Li) -> [(Rs,Ri), ...]
    for (Ls, Li), (Rs, Ri) in pairs:
        prod_targets[(Ls, Li)].append((Rs, Ri))
    for k in prod_targets:
        prod_targets[k].sort(key=lambda t: (t[0], t[1]))

    letter_iter    = iter(c for c in string.ascii_lowercase if c != RESERVED_MASK_LETTER)
    letters_per_index = {}
    L_occ_letters  = defaultdict(lambda: defaultdict(dict))
    for (Ls, Li), _ in sorted(prod_targets.items(), key=lambda x: (x[0][0], x[0][1])):
        k = max(1, left_mult.get(Ls, 1))
        q = deque()
        for occ in range(k):
            ltr = next(letter_iter)
            q.append(ltr)
            L_occ_letters[Ls][occ][Li] = ltr
        letters_per_index[(Ls, Li)] = q

    R_occ_letters = defaultdict(lambda: defaultdict(dict))
    assigned_count = Counter()
    for (Ls, Li), targets in sorted(prod_targets.items(), key=lambda x: (x[0][0], x[0][1])):
        k_left = max(1, left_mult.get(Ls, 1))
        q      = letters_per_index[(Ls, Li)]
        occ_idx = 0
        for (Rs, Ri) in targets:
            ltr   = q[occ_idx] if k_left > 1 else q[0]
            occ_idx = (occ_idx + 1) % k_left
            p_k   = max(1, right_mult.get(Rs, 1))
            p_occ = min(assigned_count[Rs], p_k - 1)
            assigned_count[Rs] += 1
            R_occ_letters[Rs][p_occ][Ri] = ltr
            key   = (Rs, Ri)
            p_occ2 = assigned_count[key] % p_k
            assigned_count[key] += 1

    L_max = defaultdict(int)
    R_max = defaultdict(int)
    for (Ls, Li), (Rs, Ri) in pairs:
        L_max[Ls] = max(L_max[Ls], Li)
        R_max[Rs] = max(R_max[Rs], Ri)

    return L_occ_letters, R_occ_letters, L_max, R_max

In [26]:
df

,#reaction_ID,reactant_IDs(atom),product_IDs(atom),reversibility
0,RBPCh,"[rb15bp[h], co2[h]]","[3pg[h], 3pg[h]]",False
1,GAPDH_nadp_hi,[3pg[h]],[dhap[h]],False
2,ALDh,"[dhap[h], e4p[h]]",[s17bp[h]],True
3,SBPase,[s17bp[h]],[s7p[h]],False
4,FBAh,"[dhap[h], dhap[h]]",[fdp_B[h]],True
...,...,...,...,...
69,GLUtm,[glu_L[c]],[glu_L[m]],True
70,ALAtm,[ala_L[c]],[ala_L[m]],True
71,PEPtm,[pep[c]],[pep[m]],True
72,CO2tm,[co2[m]],[co2[c]],True


In [27]:
def build_modreactions(df):
    rows   = ["\t".join(["#reaction.ID", "reactant.IDs(atom)", "product.IDs(atom)", "reversibility"])]

    def parse_side(text):
        return [f"{m}[{c}]" for m, c in species_pat.findall(str(text))]

    def letters_str(spec, labels):
        idxmap = labels.get(spec, {})
        return "".join(idxmap[i] for i in range(1, max(idxmap) + 1) if i in idxmap) if idxmap else ""

    def assemble_basic(specs, labels, suffix=""):
        parts = []
        for sp in specs:
            nm  = pretty(sp, suffix=suffix)
            ltr = letters_str(sp, labels)
            parts.append(f"{nm}({ltr})" if ltr else nm)
        return " + ".join(parts)

    def assemble_dups(spec_list, occ_dict, max_dict, suffix=""):
        printed = Counter()
        parts   = []
        for sp in spec_list:
            nm  = pretty(sp, suffix=suffix)
            occ = printed[sp]
            printed[sp] += 1
            letters = ""
            if sp in occ_dict and occ in occ_dict[sp]:
                idxmap  = occ_dict[sp][occ]
                letters = "".join(filter(None, (idxmap.get(i, "") for i in range(1, max_dict.get(sp, 0) + 1))))
            parts.append(f"{nm}({letters})" if letters else nm)
        return " + ".join(parts)

    for _, r in df.iterrows():
        rxn         = r["#reaction_ID"]
        left_specs  = parse_side(r["reactant_IDs(atom)"])
        right_specs = parse_side(r["product_IDs(atom)"])
        pairs       = by_rxn.get(rxn, [])

        dup_left  = any(v > 1 for v in Counter(left_specs).values())
        dup_right = any(v > 1 for v in Counter(right_specs).values())

        if pairs and (dup_left or dup_right):
            Locc, Rocc, Lmax, Rmax = build_labels_with_duplicates(pairs, left_specs, right_specs)
            left_str  = assemble_dups(left_specs,  Locc, Lmax)
            right_str = assemble_dups(right_specs, Rocc, Rmax)
            use_dups  = True
        else:
            labels    = build_labels_basic(pairs)
            left_str  = assemble_basic(left_specs,  labels)
            right_str = assemble_basic(right_specs, labels)
            use_dups  = False

        if str(rxn).lower() == "biomass":
            right_str = "biomass"
        elif not left_str and right_str:   # sink 
            left_str  = (assemble_dups(right_specs, Rocc, Rmax, suffix=".ex")
                         if use_dups else
                         assemble_basic(right_specs, labels, suffix=".ex"))
        elif left_str and not right_str:   # source 
            right_str = (assemble_dups(left_specs, Locc, Lmax, suffix=".ex")
                         if use_dups else
                         assemble_basic(left_specs, labels, suffix=".ex"))

        # --- normalise reversibility ---
        rev_raw = str(r["reversibility"]).strip().lower()
        rev = "1" if rev_raw in {"true", "1", "yes"} else "0"

        rows.append("\t".join([rxn, left_str, right_str, rev]))

    with open(OUTPUT_FILE, "w") as fh:
        fh.write("\n".join(rows))
    print(f"Wrote {OUTPUT_FILE} with {len(df)} reactions ({len(rows) - 1} rows written).")


build_modreactions(df)


Wrote modreactionz.tsv with 74 reactions (74 rows written).


In [19]:
def build_modreactions(df):
    rows = ["\t".join(["#reaction.ID", "reactant.IDs(atom)", "product.IDs(atom)", "reversibility"])]

    def parse_side(text):
        return [f"{m}[{c}]" for m, c in species_pat.findall(str(text))]

    def assemble(specs, labels, occ_dict=None, max_dict=None, suffix=""):
        printed = Counter()
        parts = []
        for sp in specs:
            nm = pretty(sp, suffix=suffix)
            occ = printed[sp]
            printed[sp] += 1

            if occ_dict and sp in occ_dict:
                idxmap = occ_dict[sp].get(occ, {})
                max_idx = max_dict.get(sp, 0)
                letters = "".join(filter(None, (idxmap.get(i, "") for i in range(1, max_idx + 1))))
            else:
                idxmap = labels.get(sp, {}) if labels else {}
                letters = "".join(idxmap[i] for i in range(1, max(idxmap) + 1) if i in idxmap) if idxmap else ""

            parts.append(f"{nm}({letters})" if letters else nm)
        return " + ".join(parts)

    for _, r in df.iterrows():
        rxn = r["#reaction_ID"]
        left_specs  = parse_side(r["reactant_IDs(atom)"])
        right_specs = parse_side(r["product_IDs(atom)"])
        pairs = by_rxn.get(rxn, [])

        has_dups = any(v > 1 for v in Counter(left_specs + right_specs).values())

        if pairs and has_dups:
            Locc, Rocc, Lmax, Rmax = build_labels_with_duplicates(pairs, left_specs, right_specs)
            asm_left  = lambda specs, suf="": assemble(specs, None, Locc, Lmax, suf)
            asm_right = lambda specs, suf="": assemble(specs, None, Rocc, Rmax, suf)
        else:
            labels = build_labels_basic(pairs)
            asm_left  = lambda specs, suf="": assemble(specs, labels, suffix=suf)
            asm_right = lambda specs, suf="": assemble(specs, labels, suffix=suf)

        left_str  = asm_left(left_specs)
        right_str = asm_right(right_specs)

        if str(rxn).lower() == "biomass":
            right_str = "biomass"
        elif not left_str and right_str:   # sink
            left_str  = asm_right(right_specs, ".ex")
        elif left_str and not right_str:   # source
            right_str = asm_left(left_specs, ".ex")

        rev_raw = str(r["reversibility"]).strip().lower()
        rev = "1" if rev_raw in {"true", "1", "yes"} else "0"
        rows.append("\t".join([rxn, left_str, right_str, rev]))

    with open(OUTPUT_FILE, "w") as fh:
        fh.write("\n".join(rows))
    print(f"Wrote {OUTPUT_FILE} with {len(df)} reactions ({len(rows) - 1} rows written).")

build_modreactions(df)

Wrote modreactionz.tsv with 74 reactions (74 rows written).
